# NCKH — V7.1 Evidence-Conflict Rescue (NO GPU) · 1.301 cặp TEST

**Mục tiêu:** thử nghiệm cải tiến V7 bằng cách định tuyến lại **trên đầu ra đã suy luận**. Không chạy lại CountGD++ / Qwen. **Không dùng `gt_count`, `pair_type`, `source_class` hoặc `split` để quyết định dự đoán.** Các cột đó chỉ dùng để đánh giá.

**Quy trình:** khôi phục 150 audit của batch 000–002 từ bundle; đọc 944 audit từ Kaggle Output; đọc 207 kết quả V7 gốc (96 positive + 111 hard-negative); kiểm tra khớp 1.301 dự đoán với V7 baseline; chạy 2 quy tắc rescue cố định; xuất MAE/FPR và thay đổi từng ảnh.

**Tệp đầu vào trên Kaggle:**
1. `V7_1_Input_Bundle.zip` (đính kèm từ câu trả lời ChatGPT; 1.301 gốc + 150 audit).
2. **`v7_AUTO_new_audits_003_021.csv`** (Kaggle Output của lần AUTO; KHÔNG phải `v7_AUTO_new_predictions_003_021.txt`).
3. `nckh_v7_dual_evidence_router_export.zip` chứa **`v7_results.csv`** (207 mẫu V7 gốc), hoặc dataset đã giải nén của nó.

Notebook dừng nếu thiếu dữ liệu hoặc phát hiện dự đoán gốc không khớp; **không tự điền số dự đoán giả**.

**Scientific status:** các quy tắc được thử sau khi đã xem một phần TEST => đây là **exploratory / hypothesis-generating**. Chỉ tuyên bố cải thiện khẳng định (confirmatory) sau đánh giá trên các cặp negative *chưa dùng để thiết kế* và positive validation/test độc lập hoặc phương pháp đánh giá minh bạch.

### Đọc file `.txt` được lưu từ Kaggle

**Không cần đổi đuôi**: notebook này nhận cả `v7_AUTO_new_audits_003_021.txt` lẫn `.csv` và tệp tương ứng nằm trong ZIP. Nội dung vẫn phải là bảng CSV có ít nhất `group_id`, `v7_count`, `baseline_count`, `v4_filtered_count`, `neg_text_count`, `neg_text_exemplar_count`, `target_evidence`, `decision`, `v7_route`, `keep_boxes_shown`, `retention_ratio`.

**Chú ý:** `v7_AUTO_new_predictions_003_021.txt` chỉ chứa ba cột `group_id,v7_count,batch_index`; nó **không phải** file audits và không đủ dữ liệu để tính V7.1. Hãy lấy đúng file `v7_AUTO_new_audits_003_021` từ Kaggle Output.



In [ ]:
# CELL 1 - CPU setup. Không cần GPU / Internet.
from pathlib import Path
import zipfile, io, json, os, re, numpy as np, pandas as pd
from IPython.display import display, FileLink

ROOTS = [Path('/kaggle/input'), Path('/kaggle/working'), Path('/mnt/data')]
OUT = Path('/kaggle/working/v7_1_rapid_eval') if Path('/kaggle').exists() else Path('/mnt/data/v7_1_rapid_eval')
OUT.mkdir(parents=True,exist_ok=True)
SEED=42
print('Output:', OUT)

# Đọc được *.csv và *.txt chứa dữ liệu CSV, kể cả file ở trong ZIP.
# Không đổi tên nội dung hay coi file predictions là audits.
def locate_csv(target_name):
    target_stem = Path(target_name).stem
    def priority(name):
        f = Path(name).name
        stem = Path(f).stem
        # chấp nhận: file.csv, file.txt, file (1).txt, file(2).csv
        if stem == target_stem:
            return 0 if f.lower().endswith('.csv') else 1
        if re.fullmatch(re.escape(target_stem) + r'\s*\(\d+\)', stem, flags=re.I):
            return 2 if f.lower().endswith('.csv') else 3
        return None

    candidates=[]
    for root in ROOTS:
        if not root.exists(): continue
        for pattern in (target_stem + '*.csv', target_stem + '*.txt'):
            for p in root.rglob(pattern):
                rank=priority(p.name)
                if rank is not None:
                    candidates.append((rank,len(str(p)),p))
    for _,__,p in sorted(candidates):
        try:
            df=pd.read_csv(p,encoding='utf-8-sig')
            if df.shape[1] < 2:
                print('Cảnh báo:',p,'không có dấu phân cách CSV hợp lệ. Cần tệp chứa các cột cách nhau bằng dấu phẩy.')
                continue
            print('Found',target_name,'as',p,'->',df.shape)
            return df
        except Exception as ex:
            print('Không đọc được',p,repr(ex))

    # Đọc file bên trong ZIP, kể cả thành viên có đuôi .txt.
    for root in ROOTS:
        if not root.exists(): continue
        for zp in sorted(root.rglob('*.zip')):
            try:
                with zipfile.ZipFile(zp) as z:
                    matches=sorted(
                        [(priority(n),n) for n in z.namelist() if priority(n) is not None],
                        key=lambda t:(t[0],len(t[1])))
                    for _,n in matches:
                        try:
                            df=pd.read_csv(io.BytesIO(z.read(n)),encoding='utf-8-sig')
                            if df.shape[1]<2: continue
                            print('Found',target_name,'inside',zp,'as',n,'->',df.shape)
                            return df
                        except Exception: pass
            except (zipfile.BadZipFile,OSError): continue
    return None

master=locate_csv('v7_official_TEST_1301_merged.csv')
a150=locate_csv('v7_audits_000_002_150.csv')
a944=locate_csv('v7_AUTO_new_audits_003_021.csv')
original=locate_csv('v7_results.csv')
need={'V7_1_Input_Bundle.zip: v7_official_TEST_1301_merged.csv':master,
      'V7_1_Input_Bundle.zip: v7_audits_000_002_150.csv':a150,
      'Kaggle Output: v7_AUTO_new_audits_003_021.csv HOẶC .txt':a944,
      'V7 original export: v7_results.csv':original}
missing=[key for key,value in need.items() if value is None]
if missing:
    raise FileNotFoundError('THIẾU FILE SAU, Add Input vào Kaggle rồi Run All lại:\n  - '+ '\n  - '.join(missing))
print('Rows:', {'master':len(master), 'a150':len(a150),'a944':len(a944),'original':len(original)})


In [ ]:
# CELL 2 — Integrity / no target leakage
assert len(master)==1301 and master.group_id.is_unique
assert len(a150)==150 and a150.group_id.is_unique
assert len(a944)==944 and a944.group_id.is_unique
assert len(original)==207 and original.group_id.is_unique
assert set(original.pair_type.astype(str))=={'POSITIVE','HARD_NEGATIVE'}
assert int((original.pair_type=='HARD_NEGATIVE').sum())==111
assert int((original.pair_type=='POSITIVE').sum())==96

required=['group_id','v7_count','baseline_count','v4_filtered_count','neg_text_count',
          'neg_text_exemplar_count','target_evidence','decision','v7_route',
          'keep_boxes_shown','retention_ratio']
chunks=[original, a150, a944]
for title,data in zip(['original207','batch000-002','auto003-021'],chunks):
    absent=[c for c in required if c not in data.columns]
    if absent: raise ValueError(f'{title} thiếu cột: {absent}. Cần AUDIT, không phải chỉ predictions.')
features=pd.concat([d[required] for d in chunks],ignore_index=True)
assert features.group_id.is_unique, 'Có group_id trùng giữa các nguồn'
assert len(features)==1301
joined=master.merge(features,on='group_id',how='left',validate='one_to_one',suffixes=('_ref',''))
assert not joined[required[1:]].isna().any().any(), 'Thiếu features ở một số group_id'
for col in ['v7_count','v7_count_ref','baseline_count','v4_filtered_count','neg_text_count',
            'neg_text_exemplar_count','keep_boxes_shown','retention_ratio']:
    joined[col]=pd.to_numeric(joined[col],errors='raise')
wrong=joined[joined.v7_count.astype(int)!=joined.v7_count_ref.astype(int)]
if len(wrong):
    display(wrong[['group_id','v7_count_ref','v7_count']].head(20))
    raise ValueError('V7 dự đoán gốc KHÔNG KHỚP file báo cáo; kiểm tra dataset/checkpoint trước khi reroute')
assert len(joined)==1301
assert (joined['gt_count']>=0).all()
print('PASS: 1.301 mẫu, dự đoán V7 gốc khớp tuyệt đối, không thiếu/duplicate.')
print(joined.pair_type.value_counts().to_string())


In [ ]:
# CELL 3 — V7.1: FROZEN inference-only rules; ABSOLUTELY NO ground-truth features here.
def reroute_v71(d):
    d=d.copy()
    d['v7_1_count']=d['v7_count'].astype(int)
    d['rescue_reason']='UNCHANGED'
    # Evidence conflict: Qwen V5 exemplar wipes plausible detections retained by V4.
    ex=(d.v7_route.astype(str).eq('NEG_TEXT_EXEMPLAR')
        & d.v4_filtered_count.ge(5)
        & d.retention_ratio.ge(0.50))
    d.loc[ex,'v7_1_count']=d.loc[ex,'v4_filtered_count'].astype(int)
    d.loc[ex,'rescue_reason']='V4_EVIDENCE_CONTRADICTS_EXEMPLAR'
    # Qwen image target STRONG + >=5 kept detections, despite strict retention threshold.
    keep=(d.v7_route.astype(str).eq('V4')
          & d.target_evidence.astype(str).str.upper().eq('STRONG')
          & d.keep_boxes_shown.ge(5)
          & d.baseline_count.gt(0))
    d.loc[keep,'v7_1_count']=d.loc[keep,'baseline_count'].astype(int)
    d.loc[keep,'rescue_reason']='STRONG_TARGET_AND_FIVE_KEEPS'
    assert (d.v7_1_count>=0).all()
    return d

final=reroute_v71(joined)
final['v7_abs_error']=(final.v7_count-final.gt_count).abs()
final['v7_1_abs_error']=(final.v7_1_count-final.gt_count).abs()
final['changed']=final.v7_count.ne(final.v7_1_count)
print('Changed:',int(final.changed.sum()),'of',len(final))
print('Changed by pair type:')
display(pd.crosstab(final.pair_type,final.rescue_reason))


In [ ]:
# CELL 4 — Paired metrics; all methods on identical samples.
def get_metrics(d,col):
    g=d.copy()
    pred=g[col].to_numpy(dtype=float); y=g['gt_count'].to_numpy(dtype=float)
    r={'n':len(g),'mae':np.abs(pred-y).mean(),'rmse':np.sqrt(np.square(pred-y).mean())}
    if g.pair_type.iloc[0]=='HARD_NEGATIVE':
        r['fpr']=(pred>0).mean()
        r['mean_negative_count']=pred.mean()
    else:r['zero_on_positive_rate']=(pred==0).mean()
    return r
rows=[]
for pair_type,sub in final.groupby('pair_type'):
    for name,col in [('V7_original','v7_count'),('V7_1_candidate','v7_1_count')]:
        rows.append({'subset':pair_type,'model':name,**get_metrics(sub,col)})
for name,col in [('V7_original','v7_count'),('V7_1_candidate','v7_1_count')]:
    p=final[col].to_numpy(float);y=final.gt_count.to_numpy(float)
    rows.append({'subset':'ALL_1301','model':name,'n':len(final),'mae':abs(p-y).mean(),'rmse':np.sqrt(((p-y)**2).mean())})
metrics=pd.DataFrame(rows)
print(metrics.to_string(index=False,float_format=lambda x:f'{x:.5f}'))
print('\nDelta MAE V7.1 − V7 (negative means improvement):')
for p,sub in final.groupby('pair_type'):
    print(p,round(float((sub.v7_1_abs_error-sub.v7_abs_error).mean()),4))

# Bootstrap paired, stratified by positive/negative (NOT CI for a new independent test)
rng=np.random.default_rng(SEED)
ci=[]
for name,sub in final.groupby('pair_type'):
    delta=(sub.v7_1_abs_error-sub.v7_abs_error).to_numpy(float)
    idx=rng.integers(0,len(delta),size=(2000,len(delta)))
    sums=delta[idx].mean(axis=1)
    ci.append({'subset':name,'delta_mae':delta.mean(), 'ci_low':np.quantile(sums,0.025), 'ci_high':np.quantile(sums,0.975)})
display(pd.DataFrame(ci))


In [ ]:
# CELL 5 — Export all results + changed cases + class stats. Label EXPLORATORY clearly.
cols=['group_id','pair_type','source_class','image_name','target','gt_count',
      'baseline_count','v4_filtered_count','target_evidence','decision','v7_route',
      'keep_boxes_shown','retention_ratio','v7_count','v7_1_count','rescue_reason','v7_abs_error','v7_1_abs_error']
final[cols].to_csv(OUT/'V7_1_EXPLORATORY_1301_predictions.csv',index=False)
final.loc[final.changed,cols].sort_values('v7_abs_error',ascending=False).to_csv(OUT/'V7_1_changed_cases.csv',index=False)
metrics.to_csv(OUT/'V7_1_EXPLORATORY_metrics.csv',index=False)
byclass=final[final.pair_type.eq('POSITIVE')].groupby('source_class').apply(
    lambda s:pd.Series({'n':len(s),'V7_MAE':s.v7_abs_error.mean(),'V7_1_MAE':s.v7_1_abs_error.mean()}),include_groups=False)
byclass.to_csv(OUT/'V7_1_positive_by_class.csv')
summary={'status':'EXPLORATORY, not confirmatory',
         'source':'V7 cached inference outputs, no additional Qwen/CountGD inference',
         'n':int(len(final)),'changed_n':int(final.changed.sum()),
         'negative_changes':int(final.loc[final.pair_type=='HARD_NEGATIVE','changed'].sum()),
         'metrics':metrics.replace({np.nan:None}).to_dict('records'),
         'warning':'Do not claim generalization using hard negatives recycled during V7 development or rule iterations viewed on TEST.'}
(OUT/'V7_1_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=float),encoding='utf-8')
import shutil
archive=shutil.make_archive(str(OUT.parent/'V7_1_EXPLORATORY_eval_1301'),'zip',root_dir=str(OUT))
print('READY',archive)
display(pd.read_csv(OUT/'V7_1_changed_cases.csv').head(30))
display(FileLink(archive))


## Báo cáo khoa học

- Gọi phương pháp **V7.1 exploratory rescue (post-hoc)** và ghi rõ đã quan sát 150 ảnh TEST khi đặt quy tắc. Đây là **phân tích khám phá**, không phải cải thiện xác nhận độc lập.
- Báo cáo **MAE positive, MAE hard-negative, hard-negative FPR, zero-on-positive, MAE chung**, cùng MAE CountGD++ trên cùng 1.190 positive nếu có predictions.
- Nếu V7.1 giảm MAE positive nhưng tăng FPR negative, **không** khẳng định cải thiện tổng thể; báo trade-off rõ ràng.
- **Bước để viết paper:** chốt V7.1, xây dựng hard-negative mới và positive held-out chưa dùng để điều chỉnh, chạy lại test một lần không chỉnh ngưỡng sau khi xem kết quả. Với CountGD baseline zero trên ảnh mật độ cực cao (ví dụ `1123.jpg`), chỉ reroute không thể cứu; cần nghiên cứu CountGD adaptive tiling với GPU và đo riêng.
